# 02 · Modelling

Three sklearn baselines vs. an entity-embedding MLP in PyTorch. Model selection on the validation split,
final numbers on the untouched test split, then a threshold chosen for campaign value rather than F1.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from churn.analysis import style
from churn.pipeline.load import read_query

style.apply()
pd.set_option("display.max_columns", 50)
%matplotlib inline

In [ ]:
from churn.config import load_config
from churn.models import evaluate as ev
from churn.models.baselines import build_models, cross_validate, logistic_coefficients
from churn.models.features import CATEGORICAL, NUMERIC, make_split

cfg = load_config()
split = make_split()
{k: len(v) for k, v in split.ids.items()}

## Baselines

In [ ]:
models = build_models(cfg["random_seed"])
cv = cross_validate(models, split.X_train, split.y_train, folds=5)
pd.DataFrame(cv).T

In [ ]:
for m in models.values():
    m.fit(split.X_train, split.y_train)

coefs = logistic_coefficients(models["logistic_regression"])
coefs.head(15)

## PyTorch tabular net
Each categorical gets an embedding of size `min(8, (cardinality + 1) // 2)`, concatenated with the
batch-normalised numerics and fed through two SiLU blocks. Early stopping on validation AUC.

In [ ]:
import torch
from churn.models.tabular_net import NetConfig, TabularNetClassifier

tc = cfg["model"]["torch"]
net = TabularNetClassifier(CATEGORICAL, NUMERIC, NetConfig(
    hidden=tc["hidden"], dropout=tc["dropout"], lr=tc["lr"], weight_decay=tc["weight_decay"],
    batch_size=tc["batch_size"], max_epochs=tc["max_epochs"], patience=tc["patience"]))
net.fit(split.X_train, split.y_train, split.X_valid, split.y_valid)
net.model

In [ ]:
fig = ev.plot_training_history(net.history, net.best_epoch)
plt.show()

### What did the embeddings learn?
The contract embedding should place month-to-month far away from the two term contracts.

In [ ]:
idx = CATEGORICAL.index("contract_type")
emb = net.model.embeddings[idx].weight.detach().cpu().numpy()
vocab = net.encoder.vocab["contract_type"]
pd.DataFrame(emb[list(vocab.values())], index=list(vocab.keys())).round(3)

## Comparison on the test split

In [ ]:
predictors = {name: (lambda m: (lambda X: m.predict_proba(X)[:, 1]))(m) for name, m in models.items()}
predictors["pytorch_tabular_net"] = net.predict_proba

test_p = {n: f(split.X_test) for n, f in predictors.items()}
valid_p = {n: f(split.X_valid) for n, f in predictors.items()}
pd.DataFrame({n: ev.classification_metrics(split.y_test, p) for n, p in test_p.items()}).T.round(4)

In [ ]:
fig = ev.plot_roc_pr(split.y_test, test_p)
plt.show()
fig = ev.plot_calibration(split.y_test, test_p)
plt.show()

## Picking a threshold
0.5 is arbitrary. With the campaign economics in `config.yaml` the threshold that maximises expected value
on validation is usually a lot lower, because missing a churner costs more than a wasted offer.

In [ ]:
best = max(valid_p, key=lambda n: ev.roc_auc_score(split.y_valid, valid_p[n]))
curve = ev.profit_curve(split.y_valid, valid_p[best], split.X_valid["monthly_charges"], cfg["economics"])
t_star = ev.best_threshold(curve)
print(best, t_star)
fig = ev.plot_profit(curve, t_star)
plt.show()

In [ ]:
ev.classification_metrics(split.y_test, test_p[best], t_star)

## What drives the prediction

In [ ]:
imp = ev.permutation_importance(predictors[best], split.X_test, split.y_test, n_repeats=5)
fig = ev.plot_importance(imp, best)
plt.show()

## Takeaways
- All four models land close together; the signal in this dataset is mostly captured by contract,
  tenure, internet type and payment method, so a well-regularised logistic regression is hard to beat.
- The embedding net needs to clearly beat the linear model to justify itself; on a 21-column snapshot the
  extra capacity has little to work with. It becomes more interesting once behavioural data
  (usage, tickets, billing history) is added.
- Contacting customers above the value-maximising threshold beats both "contact everyone" and
  "contact nobody" on the held-out test set.